_Baixar API sienge por request_


In [3]:
import os
import requests
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
user = os.getenv("POSTGRES_USER")
password = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_HOST")
port = os.getenv("POSTGRES_PORT")
db = os.getenv("POSTGRES_DB")
schema = os.getenv("POSTGRES_SCHEMA")

In [5]:
api_user = os.getenv("SIENGE_USERNAME")
api_password = os.getenv("SIENGE_PASSWORD")
url = "https://api.sienge.com.br/olimpo/public/api/v1/payment-categories"
response = requests.get(url, auth=(api_user, api_password))

In [6]:
if response.status_code == 200:
    dados = response.json()
    if dados:
        df = pd.DataFrame(dados)

        # Conexão com banco
        engine = create_engine(f"postgresql://{user}:{password}@{host}:{port}/{db}")
        df.to_sql("financialCategories", engine, schema=schema, if_exists="replace", index=False)
        print("Dados importados com sucesso!")
    else:
        print("API respondeu, mas não retornou dados.")
else:
    print("Erro ao acessar API:", response.status_code, response.text)

Dados importados com sucesso!


In [7]:
# dicionário id->name para lookup
id_map = dict(zip(df["id"].astype(str), df["name"]))

def expand_row(row):
    codigo = str(row["id"])
    # pega prefixos progressivos que existam na base
    niveis = [codigo[:i] for i in range(1, len(codigo)+1) if codigo[:i] in id_map]
    cols = {}
    for j, n in enumerate(niveis[:-1]):  # exclui o último (que é o próprio R)
        cols[f"id{j+1}"] = n           # id correspondente
        cols[f"fc{j+1}"] = id_map[n]   # nome correspondente
    
    # adiciona colunas originais da linha
    for c in row.index:
        if c == "name":
            cols["financialCategory"] = row[c]
        else: cols[c] = row[c]
    return cols

# aplica apenas para tpConta = 'R'
df_out = pd.DataFrame([expand_row(r) for _, r in df[df["tpConta"]=="R"].iterrows()])

In [ ]:
df_out.head()

,id1,fc1,id2,fc2,id3,fc3,id,financialCategory,tpConta,flRedutora,flAtiva,flAdiantamento,flImposto
0,1,ENTRADAS/RECEITAS,101,RECEITA DE INCORPORAÇÃO,10101,INCORPORAÇÕES,1010101,Receita de Venda de Unidades Imobiliárias,R,N,S,N,N
1,1,ENTRADAS/RECEITAS,101,RECEITA DE INCORPORAÇÃO,10101,INCORPORAÇÕES,1010102,Receita de Venda de Terreno,R,N,S,N,N
2,1,ENTRADAS/RECEITAS,101,RECEITA DE INCORPORAÇÃO,10101,INCORPORAÇÕES,1010103,Receita de Frações Ideais,R,N,S,N,N
3,1,ENTRADAS/RECEITAS,101,RECEITA DE INCORPORAÇÃO,10101,INCORPORAÇÕES,1010104,Receita de Unidades Vendidas em Permuta,R,N,S,N,N
4,1,ENTRADAS/RECEITAS,101,RECEITA DE INCORPORAÇÃO,10101,INCORPORAÇÕES,1010105,Receita de Vendas de imóveis adq de Terceiros,R,N,S,N,N
